# Biscuit Defect Classification — Step 0: Dataset Setup

This notebook first downloads the **Industry Biscuit (Cookie) dataset** from Kaggle.

**Goal of this project:** the goal of the project is to develop a model capable of classifying biscuits in the dataset as either normal or defective.



## 1. Install and import libraries



In [93]:
!pip install -q scikit-learn
!pip install -q kaggle

import os
from collections import Counter
import random
import numpy as np

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt



print("Libraries imported.")

Libraries imported.


## 0. Environment setup (Colab or local)

This cell lets the notebook run **both on Google Colab and on a local machine**.

- On Colab it mounts Google Drive and uses the project folder there.
- Locally it uses a `biscuit_dataset` folder next to the notebook (override with
  the `BISCUIT_DATA_DIR` environment variable if your data lives elsewhere).

Every later path is derived from `DATASET_DIR`, so you only set the location once.


In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET_DIR = "/content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset"
else:
    # Local run: default to ./biscuit_dataset, or set BISCUIT_DATA_DIR to point elsewhere.
    DATASET_DIR = os.environ.get("BISCUIT_DATA_DIR", os.path.abspath("biscuit_dataset"))

os.makedirs(DATASET_DIR, exist_ok=True)
print(f"IN_COLAB = {IN_COLAB}")
print(f"DATASET_DIR = {DATASET_DIR}")


## 2. Mount Google Drive

This connects your Google Drive to this Colab session, so any file we save
under `/content/drive/MyDrive/...` will actually persist on your Drive (Colab's
own disk is wiped every time the runtime restarts).

Running this cell will pop up a Google sign-in / permission prompt — accept it.

In [94]:
# (Google Drive mounting is now handled in the Environment setup cell above.)
# Kept as a no-op so the original notebook numbering still makes sense.


## 3. Kaggle API Setup

This cell authenticates the Kaggle API by saving the access token, setting the required file permissions, and defining the token as an environment variable. This enables the notebook to download datasets and interact with Kaggle resources.

In [95]:
KAGGLE_TOKEN = os.environ.get("KAGGLE_TOKEN", "")  # <-- replace with your real token

# Save it where the Kaggle API expects to find it
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(KAGGLE_TOKEN)
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN #In Python, os.environ[...] = ... does the same thing export does in a terminal.

print("Kaggle token saved.")

Kaggle token saved.


## 4. Download the dataset from Kaggle

This downloads the **Industry Biscuit (Cookie) dataset** as a zip file into
the Colab session's temporary storage (`/content/`).

In [96]:
!kaggle datasets download -d imonbilk/industry-biscuit-cookie-dataset -p /content --force

Dataset URL: https://www.kaggle.com/datasets/imonbilk/industry-biscuit-cookie-dataset
License(s): GNU Lesser General Public License 3.0
100% 43.5M/43.5M [00:03<00:00, 12.5MB/s]



## 5. Dataset path

Folder on Drive where the dataset will be stored.

In [97]:
# DATASET_DIR is defined in the Environment setup cell above
# (works on both Colab and local). Nothing to do here.
print(f"Dataset will be stored at: {DATASET_DIR}")


In [98]:
os.makedirs(DATASET_DIR, exist_ok=True)

print(f"Dataset will be stored at: {DATASET_DIR}")

Dataset will be stored at: /content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset


## 6. Unzip the dataset into Google Drive folder

This extracts the downloaded zip file directly into the Drive folder.

In [ ]:
import zipfile

zip_path = "/content/industry-biscuit-cookie-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATASET_DIR)

print(f"Dataset extracted to: {DATASET_DIR}")

Dataset extracted to: /content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset


## Dataset setup complete

The dataset now lives permanently at:

```
/content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset
```

## 7. Configuration



All the settings(Paths and hyperparameters) will be used for the rest of this notebook.

- `DATA_ROOT` — path to the dataset folder on your Drive (adjust if your
  folder name/location is different).
- `IMG_SIZE` — every image gets resized to this size (128x128) before being
  fed to the model. Keeps input shape consistent and keeps training fast.
- `BATCH_SIZE` — how many images are processed together in one training step.
- `VAL_RATIO` / `TEST_RATIO` — 15% of the data goes to validation, 15% to
  test, the remaining 70% is used for training.
- `SEED` — fixes randomness so your split is reproducible every time you
  rerun this notebook.

In [99]:
DATA_ROOT  = os.path.join(DATASET_DIR, "IndustryBiscuit")
IMAGES_DIR = os.path.join(DATA_ROOT, "Images")
CSV_PATH   = os.path.join(DATA_ROOT, "Annotations.csv")

IMG_SIZE   = 128
BATCH_SIZE = 64
VAL_RATIO  = 0.15
TEST_RATIO = 0.15
SEED       = 42

print(f"Images dir: {IMAGES_DIR}")
print(f"CSV path:   {CSV_PATH}")


Images dir: /content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset/IndustryBiscuit/Images
CSV path:   /content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset/IndustryBiscuit/Annotations.csv


## 8. Load labels from Annotations.csv

The dataset's CSV has 3 columns: `file`, `classDescription`, `classCode`.
There are 4 fine-grained classes in `classDescription`:

- `Defect_No` (normal biscuit)
- `Defect_Shape`, `Defect_Object`, `Defect_Color` (three different kinds of
  defects)

Since our project only needs **binary** classification (normal vs.
defective), we collapse the 3 defect types into a single "defective" label:

- `Defect_No` &rarr; label `0` (normal)
- anything else &rarr; label `1` (defective)

We also build the full file path for each image, since the CSV only stores
the filename, not the folder.

In [100]:
from IPython.display import display

def load_labels():

    df = pd.read_csv(CSV_PATH)
    print("Before adjusting lables:\n")
    display(df.sample(20, random_state=SEED))
    print("\n")
    # 0 = normal, 1 = defective (any defect type)
    df["label"] = (df["classDescription"] != "Defect_No").astype(int)
    df["path"]  = df["file"].apply(lambda f: os.path.join(IMAGES_DIR, f))
    return df[["path", "label"]]


"""def filter_unreadable_images(df):

    Some real-world datasets contain a handful of corrupted or invalid image
    files (wrong format, truncated download, etc). Rather than crash deep
    into training when the DataLoader happens to hit one, we check every
    file ONCE up front with PIL's verify(), and drop any that fail.
    This is slower the first time (has to open every file) but makes the
    rest of the notebook robust -- training will never crash on a bad image.

    good_rows = []
    bad_files = []

    for i, (_, row) in enumerate(df.iterrows()):
        if i % 500 == 0:
            print(f"  checked {i}/{len(df)}...")
        try:
            with Image.open(row["path"]) as img:
                img.verify()
            good_rows.append(row)
        except Exception:
            bad_files.append(row["path"])

    clean_df = pd.DataFrame(good_rows).reset_index(drop=True)

    if bad_files:
        print(f"Found {len(bad_files)} unreadable image(s), excluding them:")
        for f in bad_files:
            print(f"  - {f}")
    else:
        print("No unreadable images found.")

    return clean_df"""


labels_df = load_labels()
print("After adjusting lables:\n")
display(labels_df.sample(20, random_state=42))
print(f"\nTotal images: {len(labels_df)}")
print(f"Class counts: {dict(Counter(labels_df['label']))}  (0=normal, 1=defective)")


Before adjusting lables:



          file classDescription  classCode
4657  4658.jpg     Defect_Shape          1
3539  3540.jpg     Defect_Shape          1
907   0908.jpg     Defect_Shape          1
4353  4354.jpg        Defect_No          0
3745  3746.jpg     Defect_Shape          1
4633  4634.jpg     Defect_Shape          1
2244  2245.jpg        Defect_No          0
1924  1925.jpg        Defect_No          0
3802  3803.jpg     Defect_Shape          1
2634  2635.jpg        Defect_No          0
2827  2828.jpg        Defect_No          0
79    0080.jpg     Defect_Shape          1
350   0351.jpg        Defect_No          0
1370  1371.jpg     Defect_Shape          1
4481  4482.jpg        Defect_No          0
3784  3785.jpg     Defect_Shape          1
2890  2891.jpg        Defect_No          0
3193  3194.jpg    Defect_Object          2
1281  1282.jpg     Defect_Shape          1
4726  4727.jpg     Defect_Shape          1



After adjusting lables:



                                                   path  label
4657  /content/drive/MyDrive/Colab_Notebooks/vision_...      1
3539  /content/drive/MyDrive/Colab_Notebooks/vision_...      1
907   /content/drive/MyDrive/Colab_Notebooks/vision_...      1
4353  /content/drive/MyDrive/Colab_Notebooks/vision_...      0
3745  /content/drive/MyDrive/Colab_Notebooks/vision_...      1
4633  /content/drive/MyDrive/Colab_Notebooks/vision_...      1
2244  /content/drive/MyDrive/Colab_Notebooks/vision_...      0
1924  /content/drive/MyDrive/Colab_Notebooks/vision_...      0
3802  /content/drive/MyDrive/Colab_Notebooks/vision_...      1
2634  /content/drive/MyDrive/Colab_Notebooks/vision_...      0
2827  /content/drive/MyDrive/Colab_Notebooks/vision_...      0
79    /content/drive/MyDrive/Colab_Notebooks/vision_...      1
350   /content/drive/MyDrive/Colab_Notebooks/vision_...      0
1370  /content/drive/MyDrive/Colab_Notebooks/vision_...      1
4481  /content/drive/MyDrive/Colab_Notebooks/vision_...


Total images: 4900
Class counts: {1: 3004, 0: 1896}  (0=normal, 1=defective)


## 9. Stratified train / validation / test split

"Stratified" means each split (train, val, test) keeps the **same ratio** of
normal-to-defective images as the full dataset. Without stratification, a
random split could accidentally put too many defective images in one split
and too few in another, skewing your results.

We do this in two steps:
1. Split off the training set (70%) from a combined val+test set (30%).
2. Split that combined 30% into validation (15%) and test (15%).

`stratify=df["label"]` is what tells `train_test_split` to preserve the class
ratio in both steps.

In [101]:
def split_dataset(df):
    # Step 1: train vs (val + test)
    train_df, temp_df = train_test_split(
        df,
        test_size=VAL_RATIO + TEST_RATIO,
        stratify=df["label"],
        random_state=SEED,
    )
    # Step 2: val vs test (split the temp set in half, proportionally)
    rel_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=rel_test,
        stratify=temp_df["label"],
        random_state=SEED,
    )
    return train_df, val_df, test_df

train_df, val_df, test_df = split_dataset(labels_df)

print(f"Train: {len(train_df):>4}   class counts: {dict(Counter(train_df['label']))}")
print(f"Val:   {len(val_df):>4}   class counts: {dict(Counter(val_df['label']))}")
print(f"Test:  {len(test_df):>4}   class counts: {dict(Counter(test_df['label']))}")

Train: 3430   class counts: {0: 1327, 1: 2103}
Val:    735   class counts: {1: 450, 0: 285}
Test:   735   class counts: {1: 451, 0: 284}


## 10. Define the PyTorch Dataset class

A PyTorch `Dataset` needs two methods:
- `__len__` — tells PyTorch how many samples exist
- `__getitem__` — given an index, loads and returns that one sample (image +
  label)

We open each image with PIL, convert it to RGB (in case any image is
grayscale or has an alpha channel), apply our preprocessing/augmentation
(`transform`), and return it along with its label as a float (since we'll
use a loss function — `BCEWithLogitsLoss` — that expects float labels).

In [102]:
class BiscuitDataset(Dataset):
    def __init__(self, df, transform):
        self.paths  = df["path"].tolist()
        self.labels = df["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except Exception:
            # Defensive fallback: if a bad file somehow still slipped through
            # (e.g. you skipped the filtering step), substitute a different
            # sample instead of crashing the whole training run.

            fallback_idx = (idx + 1) % len(self.paths) #Defensive Programming
            img = Image.open(self.paths[fallback_idx]).convert("RGB")
            return self.transform(img), float(self.labels[fallback_idx])
        return self.transform(img), float(self.labels[idx])

## 11. Define preprocessing and augmentation transforms

Two separate pipelines:

**Training transform** (with augmentation): randomly flips, rotates, and
mildly jitters brightness/contrast on each image every time it's loaded.
This artificially increases the diversity of the training data and helps
prevent overfitting, since the model sees slightly different versions of
each image across epochs.

We keep `ColorJitter` mild (only 10% brightness/contrast change) because one
of our defect types is literally `Defect_Color` — if we jitter colors too
aggressively, we risk washing out the very feature the model needs to learn
for that class.

**Evaluation transform** (no augmentation): used for validation and test
sets. We want validation/test scores to reflect the model's real performance
on unmodified images, not on randomly altered ones.

Both pipelines resize to `IMG_SIZE x IMG_SIZE`, convert to a PyTorch tensor,
and normalize pixel values to roughly [-1, 1] (mean 0.5, std 0.5 per
channel) — a standard preprocessing choice that helps training converge
faster.

In [103]:
_normalize = T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    _normalize,
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    _normalize,
])

print("Transforms defined.")

Transforms defined.


## 12. Build the Dataset objects and DataLoaders

We wrap each split (train/val/test) in a `BiscuitDataset`, using the
augmenting transform for training and the plain transform for val/test.

Then we wrap each dataset in a `DataLoader`, which handles batching
(grouping images into batches of `BATCH_SIZE`), shuffling (only for
training — val/test order doesn't matter), and parallel loading
(`num_workers=2` loads images in the background while the GPU is busy
training, instead of one at a time).

In [104]:
train_ds = BiscuitDataset(train_df, train_transform)
val_ds   = BiscuitDataset(val_df,   eval_transform)
test_ds  = BiscuitDataset(test_df,  eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 54
Val batches:   12
Test batches:  12


## 13. Compute the class weight for the loss function

Our classes are mildly imbalanced (~39% normal, ~61% defective in the full
dataset). Without correction, the model could get decent accuracy just by
leaning toward predicting "defective" more often.

We fix this with `pos_weight` for `BCEWithLogitsLoss` (the loss function
we'll use in training):

```
pos_weight = (number of normal images) / (number of defective images)
```

Since defective images are the majority class here, `pos_weight` comes out
to less than 1 — it slightly *down-weights* defective errors and
*up-weights* normal-class errors relative to a plain unweighted loss,
balancing out the fact there are more defective examples to learn from.

In [105]:
n_norm = (train_df["label"] == 0).sum()
n_def  = (train_df["label"] == 1).sum()
pos_weight = torch.tensor([n_norm / n_def], dtype=torch.float32)

print(f"Normal in train:     {n_norm}")
print(f"Defective in train:  {n_def}")
print(f"pos_weight for BCE loss: {pos_weight.item():.3f}")

Normal in train:     1327
Defective in train:  2103
pos_weight for BCE loss: 0.631


## 14. Pull one batch and inspect it

Before moving on, let's confirm everything works end-to-end: grab one batch
from the training loader and check its shape and value range.



In [106]:
images, labels = next(iter(train_loader))

print(f"Batch images shape: {tuple(images.shape)}  (batch, channels, H, W)")
print(f"Batch labels (first 8): {labels[:8].tolist()}")
print(f"Pixel value range: [{images.min():.2f}, {images.max():.2f}]")
print("\nDataset preparation complete.")

Batch images shape: (64, 3, 128, 128)  (batch, channels, H, W)
Batch labels (first 8): [1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0]
Pixel value range: [-1.00, 1.00]

Dataset preparation complete.


Lets see how 8th image from first batch actually look like

In [107]:

single_img = images[7]
single_label = labels[0].item()
single_img = single_img * 0.5 + 0.5
single_img = single_img.permute(1, 2, 0)
single_img_numpy = single_img.numpy()


plt.imshow(single_img_numpy)
plt.title(f"Label: {int(single_label)} ({'Defective' if single_label == 1.0 else 'Normal'})")
plt.axis("off")
plt.show()

<Figure size 640x480 with 1 Axes>

---

# Part 2: Model definition

Now we define the CNN itself — built entirely from scratch using PyTorch's
basic building blocks (`nn.Conv2d`, `nn.BatchNorm2d`, etc.), with no
pretrained weights anywhere, per the project's requirement.

**Architecture overview:**

```
Input (3 x 128 x 128 RGB image)
  |
Conv block 1  (3   -> 32  filters)   -> 32  x 64 x 64
Conv block 2  (32  -> 64  filters)   -> 64  x 32 x 32
Conv block 3  (64  -> 128 filters)   -> 128 x 16 x 16
Conv block 4  (128 -> 128 filters)   -> 128 x  8 x  8
  |
Global Average Pooling                -> 128
  |
Dense head: FC(128->64) -> ReLU -> Dropout(0.5) -> FC(64->1)
  |
Output: 1 raw logit (apply sigmoid to get P(defective))
```

Each conv block does: `Conv2d(3x3, padding=1) -> BatchNorm2d -> ReLU ->
MaxPool2d(2x2)`. The conv layer extracts patterns, BatchNorm stabilizes
training, ReLU adds non-linearity, and MaxPool shrinks the spatial size
while keeping the strongest signals — halving width and height at each
block while doubling (then holding steady) the channel depth.

## 15. Define the ConvBlock building block

Rather than repeating the same 4 lines (Conv, BatchNorm, ReLU, MaxPool) four
times, we wrap them in a small reusable PyTorch module. Each `ConvBlock`
takes a number of input channels and output channels, and internally:

1. `Conv2d` — slides `out_channels` learnable 3x3 filters across the input.
   `padding=1` keeps the spatial size unchanged at this step (only MaxPool
   shrinks it).
2. `BatchNorm2d` — normalizes the activations of each channel across the
   batch, stabilizing and speeding up training.
3. `ReLU` — zeroes out negative values, adding the non-linearity that lets
   the network learn complex patterns.
4. `MaxPool2d(2x2)` — halves the spatial dimensions by keeping only the
   maximum value in each 2x2 window, which also gives a little tolerance to
   small shifts in defect position.

In [108]:
class ConvBlock(nn.Module):
    """Conv -> GroupNorm -> ReLU -> MaxPool. Halves spatial size, expands channels.

    GroupNorm (instead of BatchNorm) has no running statistics, so it behaves
    identically in train() and eval() mode. This removes the BatchNorm
    eval-mode instability that was causing the validation-accuracy spikes.
    """

    def __init__(self, in_channels, out_channels, num_groups=8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        # num_groups must divide out_channels. 8 divides 32, 64, and 128 cleanly.
        self.norm = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x)
        x = self.relu(x)
        x = self.pool(x)
        return x

print("ConvBlock (GroupNorm) defined.")

ConvBlock (GroupNorm) defined.


## 16. Define the full BiscuitCNN model

Now we stack four `ConvBlock`s, then add:

- **Global Average Pooling** (`nn.AdaptiveAvgPool2d(1)`) — instead of
  flattening the final `128 x 8 x 8` feature map into a giant 8192-length
  vector (which would need a huge, overfit-prone dense layer), we average
  each of the 128 channels down to a single number. This collapses
  `128 x 8 x 8` into just `128` values, with zero extra parameters.
- **Dense head** — a small fully-connected network: `128 -> 64` (with ReLU),
  then heavy `Dropout(0.5)1234`

We don't apply a sigmoid inside the model — we'll do that separately when
interpreting predictions, because the loss function we'll use during
training (`BCEWithLogitsLoss`) expects raw logits and applies the sigmoid
internally (this is more numerically stable than doing it manually).

In [109]:
class BiscuitCNN(nn.Module):
    """Custom 4-block CNN for binary biscuit defect classification."""

    def __init__(self, dropout_rate: float = 0.5):
        super().__init__()

        # 4 conv blocks. Channels grow:    3 -> 32 -> 64 -> 128 -> 128
        # Spatial halves at each block:  128 -> 64 -> 32 -> 16  ->  8
        self.block1 = ConvBlock(3,   32)
        self.block2 = ConvBlock(32,  64)
        self.block3 = ConvBlock(64,  128)
        self.block4 = ConvBlock(128, 128)

        # Global Average Pooling: (B, 128, 8, 8) -> (B, 128, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Dense head producing a single logit.
        self.head = nn.Sequential(
            nn.Flatten(),                  # (B, 128, 1, 1) -> (B, 128)
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),              # raw logit
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = self.head(x)
        return x.squeeze(1)  # (B, 1) -> (B,) -- one logit per image

print("BiscuitCNN defined.")

BiscuitCNN defined.


## 17. Instantiate the model and inspect it

Let's create an instance of the model and check two things before training:

1. **Parameter count** — how many learnable numbers (weights/biases) the
   model has in total. This tells us how "big" the model is. A count in the
   hundreds of thousands is appropriately small for a from-scratch model on
   ~3,500 training images (too many parameters relative to data would risk
   overfitting).
2. **A dummy forward pass** — we feed in a fake batch of random noise shaped
   like our real images, just to confirm the model runs end-to-end without
   shape errors, and that the output shape matches what we expect (one
   number per image in the batch).

In [110]:
model = BiscuitCNN()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")

# Dummy forward pass with random noise, same shape as a real batch
dummy = torch.randn(4, 3, IMG_SIZE, IMG_SIZE)
out = model(dummy)
print(f"\nDummy input  : {tuple(dummy.shape)}")
print(f"Dummy output : {tuple(out.shape)}  (one logit per image)")

Total parameters     : 249,857
Trainable parameters : 249,857

Dummy input  : (4, 3, 128, 128)
Dummy output : (4,)  (one logit per image)


## 18. Run the model on a real batch from our DataLoader

As a final check, let's pass an actual batch of biscuit images (the one we
loaded back in step 10) through the model we just built, end-to-end. This
confirms the dataset pipeline and the model are fully compatible with each
other before we move on to writing the training loop.


In [111]:
real_output = model(images)
print(f"Real batch input shape:  {tuple(images.shape)}")
print(f"Real batch output shape: {tuple(real_output.shape)}")
print(f"Sample raw logits (untrained model): {real_output[:8].detach().tolist()}")

print("\nModel and data pipeline are compatible. Ready for training (next notebook).")

Real batch input shape:  (64, 3, 128, 128)
Real batch output shape: (64,)
Sample raw logits (untrained model): [0.7301673889160156, 0.34388700127601624, 0.14847388863563538, 0.5559087991714478, 0.35448548197746277, 0.12248900532722473, -0.10464643687009811, 0.5339316129684448]

Model and data pipeline are compatible. Ready for training (next notebook).


## Data and model ready

Now:
- A working data pipeline (`train_loader`, `val_loader`, `test_loader`,
  `pos_weight`) that reads labels from `Annotations.csv` and applies the
  right preprocessing/augmentation.
- A custom CNN (`model`) built entirely from scratch, with no pretrained
  weights.



---

# Part 3: Training the model

Now we train `BiscuitCNN` on the biscuit dataset. This section covers:

1. Reproducibility (seeding) and device setup (GPU vs CPU)
2. Loss function and optimizer
3. The training loop itself (forward pass, backward pass, weight update)
4. Validation tracking after each epoch
5. Saving the best model automatically (based on validation performance)
6. Plotting training curves



## 19. Set the random seed and pick the device


**Device**: `torch.cuda.is_available()` checks whether a GPU is attached to
this Colab session. If yes, we move the model and data to the GPU (`cuda`),
which is dramatically faster for the matrix-heavy operations in a CNN. If
no GPU is available, we fall back to CPU automatically — slower, but still
correct.

In [112]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Go to Runtime > Change runtime type > T4 GPU for faster training.")

Using device: cuda
GPU: Tesla T4


## 20. Move the model to the device, define the loss function and optimizer

- **`model.to(device)`** — moves all the model's weights to the GPU (or
  keeps them on CPU if no GPU is available). The data from the DataLoader
  also needs to be moved to the same device at training time, or PyTorch
  will throw an error (CPU and GPU tensors can't interact directly).

- **Loss function: `BCEWithLogitsLoss`** — the standard choice for binary
  classification with a single output logit. It combines a sigmoid
  activation and binary cross-entropy loss into one numerically stable
  operation. We pass in `pos_weight` (computed back in step 13) so the loss
  accounts for our mild class imbalance.

- **Optimizer: `Adam`** — updates the model's weights after each batch,
  using the gradients computed during backpropagation. Adam adapts the
  learning rate per-parameter automatically, which makes it a robust,
  low-maintenance default for training from scratch (compared to plain SGD,
  which usually needs more manual tuning).

- **`learning_rate = 3e-4`** — a standard starting point for Adam. Controls
  how big a step the optimizer takes when updating weights. Too high and
  training becomes unstable; too low and training takes forever.

In [113]:
model = model.to(device)
pos_weight = pos_weight.to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

learning_rate = 3e-4
NUM_EPOCHS = 20  # define here so the scheduler knows the horizon
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-4,            # mild L2 regularization -> more stable
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
print(f"Optimizer: Adam, lr={learning_rate}, weight_decay=1e-4")
print("\n")
print(f"Scheduler: CosineAnnealingLR over {NUM_EPOCHS} epochs")

Optimizer: Adam, lr=0.0003, weight_decay=1e-4


Scheduler: CosineAnnealingLR over 20 epochs


## 21. Define one training epoch and one validation epoch as functions

We separate the logic into two reusable functions, since we'll call them
once per epoch in the main training loop below.

**`train_one_epoch`** — for each batch of images:
1. Move images/labels to the device (GPU/CPU)
2. `optimizer.zero_grad()` — clear gradients from the previous batch (they
   accumulate by default in PyTorch, so we must reset them each step)
3. Forward pass: feed images through the model to get predictions (logits)
4. Compute the loss (how wrong the predictions were)
5. `loss.backward()` — backpropagation: compute how much each weight
   contributed to the error
6. `optimizer.step()` — update the weights using those gradients

We track the running loss and accuracy across all batches to report at the
end of the epoch. `model.train()` puts the model in training mode (so
Dropout and BatchNorm behave correctly — Dropout randomly zeroes neurons,
BatchNorm uses batch statistics).

**`evaluate`** — same idea, but:
- No `zero_grad`, `backward`, or `optimizer.step()` — we're not updating
  weights, just measuring performance
- `torch.no_grad()` — disables gradient tracking, since we don't need
  gradients for evaluation (saves memory and computation)
- `model.eval()` — switches Dropout off and makes BatchNorm use its learned
  running statistics instead of the current batch's statistics, which is
  the correct behavior at evaluation time

This same `evaluate` function will be reused for validation during training
AND for the final test set evaluation later.

In [114]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()


        # Gradient clipping: caps the magnitude of weight updates so one
        # unlucky batch can't push the model into a bad region and corrupt
        # the BatchNorm running statistics (the cause of the val spikes).
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)


        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

print("Training and evaluation functions defined.")

Training and evaluation functions defined.


## 22. The main training loop

This runs for `NUM_EPOCHS` epochs. One epoch = one full pass through the
entire training set. For each epoch:

1. Train on the full training set (one call to `train_one_epoch`)
2. Evaluate on the validation set (one call to `evaluate`)
3. Print both losses/accuracies so we can watch progress
4. **If this epoch's validation accuracy is the best so far, save the
   model's weights to disk.** This is important: the model's performance
   can fluctuate slightly epoch to epoch, and the *last* epoch isn't always
   the *best* one. By checkpointing only when validation accuracy improves,
   we end up keeping the best version of the model, not just whatever it
   looked like after the final epoch.

We save weights in the Google Drive folder (not just Colab's temporary
disk), so the trained model survives after this Colab session ends and we
never have to retrain from scratch to use it again later.

`NUM_EPOCHS = 20` is a reasonable starting point for a dataset and model
this size — small CNNs on ~3,500 training images typically converge well
within 15-30 epochs.

In [115]:
MODEL_SAVE_PATH = os.path.join(DATA_ROOT, "..", "biscuit_cnn_best.pth")
MODEL_SAVE_PATH = os.path.normpath(MODEL_SAVE_PATH)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:>2}/{NUM_EPOCHS} | "
          f"train_loss: {train_loss:.4f}  train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> New best model saved (val_acc={val_acc:.4f})")

print(f"\nTraining complete. Best validation accuracy: {best_val_acc:.4f}")
print(f"Best model weights saved to: {MODEL_SAVE_PATH}")

Epoch  1/20 | train_loss: 0.4621  train_acc: 0.6621 | val_loss: 0.3802  val_acc: 0.6980
  -> New best model saved (val_acc=0.6980)
Epoch  2/20 | train_loss: 0.2987  train_acc: 0.8388 | val_loss: 0.1728  val_acc: 0.9075
  -> New best model saved (val_acc=0.9075)
Epoch  3/20 | train_loss: 0.1677  train_acc: 0.9289 | val_loss: 0.1322  val_acc: 0.9184
  -> New best model saved (val_acc=0.9184)
Epoch  4/20 | train_loss: 0.1203  train_acc: 0.9499 | val_loss: 0.1543  val_acc: 0.9143
Epoch  5/20 | train_loss: 0.1221  train_acc: 0.9469 | val_loss: 0.1655  val_acc: 0.9116
Epoch  6/20 | train_loss: 0.0988  train_acc: 0.9551 | val_loss: 0.1072  val_acc: 0.9469
  -> New best model saved (val_acc=0.9469)
Epoch  7/20 | train_loss: 0.0874  train_acc: 0.9606 | val_loss: 0.0564  val_acc: 0.9796
  -> New best model saved (val_acc=0.9796)
Epoch  8/20 | train_loss: 0.0908  train_acc: 0.9606 | val_loss: 0.0639  val_acc: 0.9646
Epoch  9/20 | train_loss: 0.0507  train_acc: 0.9781 | val_loss: 0.0516  val_acc: 

## 23. Plot training curves

Visualizing loss and accuracy over epochs helps diagnose how training went.



In [116]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train loss")
axes[0].plot(history["val_loss"], label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss over epochs")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["train_acc"], label="Train accuracy")
axes[1].plot(history["val_acc"], label="Val accuracy")
axes[1].axhline(y=0.90, color="red", linestyle="--", alpha=0.5, label="90% target")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy over epochs")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(MODEL_SAVE_PATH), "training_curves.png"), dpi=150)
plt.show()

<Figure size 1200x400 with 2 Axes>

---

# Part 4: Final evaluation on the test set

So far we've only looked at validation performance (used during training to
pick the best checkpoint). Now we evaluate on the **test set** — data the
model has never influenced any decision on (not training, not checkpoint
selection) — which gives the most honest estimate of real-world
performance.

We also go beyond plain accuracy here. **Why this matters for an imbalanced
dataset like ours:** accuracy alone can be misleading. For example, if a
model just predicted "defective" for every single image, it would still get
~61% accuracy on this dataset (since ~61% of images really are defective) —
without learning anything useful. Precision, recall, F1, and the confusion
matrix reveal whether the model is actually distinguishing the two classes
correctly, not just exploiting the class imbalance.

## 24. Load the best saved model weights

Before final evaluation, we reload the **best** checkpoint we saved during
training (step 22) — not necessarily the model state from the last epoch.
This ensures we're reporting results for the best version of the model,
even if performance fluctuated near the end of training.

This is also a good demonstration that the save/load mechanism works: in
any future session, you could skip training entirely and just run this cell
(plus the setup cells above it) to get a working model instantly.

In [117]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model = model.to(device)
model.eval()

print(f"Loaded best model weights from: {MODEL_SAVE_PATH}")

Loaded best model weights from: /content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset/biscuit_cnn_best.pth


## 25. Compute test set predictions and metrics

We run the model over the entire test set once, collecting:
- **Accuracy** — fraction of correct predictions overall.
- **Precision** — of all images the model called "defective", what fraction
  actually were defective? (High precision = few false alarms.)
- **Recall** — of all images that actually were defective, what fraction did
  the model catch? (High recall = few missed defects — usually the more
  important metric in quality control, since missing a real defect is
  costlier than a false alarm.)
- **F1 score** — the harmonic mean of precision and recall, a single number
  balancing both.
- **Confusion matrix** — a table showing exactly how many normal/defective
  images were classified correctly vs. incorrectly, broken down by type of
  mistake.

We use `scikit-learn`'s metric functions rather than writing these
calculations by hand, since they're standard, well-tested implementations.

In [118]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = (torch.sigmoid(logits) >= 0.5).float().cpu()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds)
test_recall = recall_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds)

print(f"Test accuracy:  {test_acc:.4f}")
print(f"Test precision: {test_precision:.4f}  (of predicted defective, % actually defective)")
print(f"Test recall:    {test_recall:.4f}  (of actual defective, % correctly caught)")
print(f"Test F1 score:  {test_f1:.4f}")

if test_acc >= 0.90:
    print(f"\nTarget met: {test_acc:.1%} accuracy exceeds the 90% requirement.")
else:
    print(f"\nBelow target: {test_acc:.1%} accuracy is under the 90% requirement. Consider more epochs or tuning.")

print("\nFull classification report:")
print(classification_report(all_labels, all_preds, target_names=["normal", "defective"]))

Test accuracy:  0.9973
Test precision: 1.0000  (of predicted defective, % actually defective)
Test recall:    0.9956  (of actual defective, % correctly caught)
Test F1 score:  0.9978

Target met: 99.7% accuracy exceeds the 90% requirement.

Full classification report:
              precision    recall  f1-score   support

      normal       0.99      1.00      1.00       284
   defective       1.00      1.00      1.00       451

    accuracy                           1.00       735
   macro avg       1.00      1.00      1.00       735
weighted avg       1.00      1.00      1.00       735



## 26. Visualize the confusion matrix

A confusion matrix lays out predictions in a 2x2 grid:

```
                  Predicted normal   Predicted defective
Actual normal      True Negative      False Positive
Actual defective   False Negative     True Positive
```

This makes it immediately visible **what kind** of mistakes the model
makes, not just how many. For instance, a high false-negative count would
mean the model is missing real defects — important to know in a quality
control context, since that's the costlier error type.

In [119]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Normal", "Defective"])
ax.set_yticklabels(["Normal", "Defective"])
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion matrix (test set)")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(MODEL_SAVE_PATH), "confusion_matrix.png"), dpi=150)
plt.show()

<Figure size 500x400 with 2 Axes>

## 27. Inspect a few sample predictions

Numbers are useful, but seeing actual examples builds intuition for what the
model gets right and wrong. We display a small grid of test images with
their true label and the model's prediction, color-coding correct
predictions green and incorrect ones red.

In [120]:
import torchvision

def denormalize(img_tensor):
    # Reverses the (mean=0.5, std=0.5) normalization for display purposes.
    img = img_tensor.clone().cpu()
    img = img * 0.5 + 0.5
    return img.clamp(0, 1)

images, labels = next(iter(test_loader))
images_device = images.to(device)
with torch.no_grad():
    logits = model(images_device)
    probs = torch.sigmoid(logits).cpu()
    preds = (probs >= 0.5).float()

n_show = 8
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis("off")
        continue
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    true_label = "Defective" if labels[i].item() == 1 else "Normal"
    pred_label = "Defective" if preds[i].item() == 1 else "Normal"
    confidence = probs[i].item() if preds[i].item() == 1 else 1 - probs[i].item()
    correct = labels[i].item() == preds[i].item()

    ax.imshow(img)
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.0%})",
                 color="green" if correct else "red", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(MODEL_SAVE_PATH), "sample_predictions.png"), dpi=150)
plt.show()

<Figure size 1400x700 with 8 Axes>



# Part 5: Project metadata

In [ ]:
metadata = {
    "project_title": "Anomaly Detection: Classifying Biscuit Images as Normal or Defective",
    "dataset_name": "Industry Biscuit Cookie Dataset (Kaggle)",
    "dataset_source": "https://www.kaggle.com/datasets/imonbilk/industry-biscuit-cookie-dataset",
    "task_type": "binary_image_classification",
    "constraint": "Model built and trained from scratch -- no pretrained weights or prepared architectures",
    "methods": ["custom_cnn_4_block"],
    "evaluation_metrics": ["accuracy", "precision", "recall", "f1_score", "confusion_matrix"],
    "accuracy_target": "> 90%",
    "team_members": [
        {"name": "Seyed Nima Ahmadi", "student_id": "948413", "role": "Full pipeline: data, model, training, evaluation"},
    ],
    "submission_date": "",
}

for key, value in metadata.items():
    print(f"{key}: {value}")

project_title: Anomaly Detection: Classifying Biscuit Images as Normal or Defective
dataset_name: Industry Biscuit Cookie Dataset (Kaggle)
dataset_source: https://www.kaggle.com/datasets/imonbilk/industry-biscuit-cookie-dataset
task_type: binary_image_classification
constraint: Model built and trained from scratch -- no pretrained weights or prepared architectures
methods: ['custom_cnn_4_block']
evaluation_metrics: ['accuracy', 'precision', 'recall', 'f1_score', 'confusion_matrix']
accuracy_target: > 90%
team_members: [{'name': 'Seyed Nima Ahmadi', 'student_id': '948413', 'role': 'Full pipeline: data, model, training, evaluation'}]
submission_date: 


## 28. Environment and reproducibility info



In [ ]:
import sys
import torch
import torchvision
import sklearn

print(f"Python version:      {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Torchvision version:  {torchvision.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:                  {torch.cuda.get_device_name(0)}")
print(f"Random seed used:     {SEED}")

Python version:      3.12.13
PyTorch version:      2.11.0+cu128
Torchvision version:  0.26.0+cu128
scikit-learn version: 1.6.1
CUDA available:       True
GPU:                  Tesla T4
Random seed used:     42


## 29. Final results summary



In [ ]:
print("=" * 50)
print("FINAL RESULTS SUMMARY")
print("=" * 50)
print(f"Model:              Custom 4-block CNN (from scratch)")
print(f"Total parameters:   {sum(p.numel() for p in model.parameters()):,}")
print(f"Training epochs:    {NUM_EPOCHS}")
print(f"Best val accuracy:  {best_val_acc:.4f}")
print(f"Test accuracy:      {test_acc:.4f}")
print(f"Test precision:     {test_precision:.4f}")
print(f"Test recall:        {test_recall:.4f}")
print(f"Test F1 score:      {test_f1:.4f}")
print(f"Target (>90% acc):  {'MET' if test_acc >= 0.90 else 'NOT MET'}")
print(f"Saved weights at:   {MODEL_SAVE_PATH}")
print("=" * 50)

FINAL RESULTS SUMMARY
Model:              Custom 4-block CNN (from scratch)
Total parameters:   249,857
Training epochs:    20
Best val accuracy:  0.9946
Test accuracy:      0.9891
Test precision:     0.9911
Test recall:        0.9911
Test F1 score:      0.9911
Target (>90% acc):  MET
Saved weights at:   /content/drive/MyDrive/Colab_Notebooks/vision_for_insdusrty_project/biscuit_dataset/biscuit_cnn_best.pth


## Project complete

**What this notebook covers, end to end:**
1. Dataset download and setup (Kaggle to Google Drive)
2. Dataset preparation: label extraction, stratified train/val/test split,
   PyTorch `Dataset`/`DataLoader`, class imbalance handling
3. A custom CNN built entirely from scratch (no pretrained weights)
4. Training with validation tracking and automatic best-model checkpointing
5. Full evaluation: accuracy, precision, recall, F1, confusion matrix, and
   visual inspection of sample predictions
